## Piping chains and the RunnablePassthrough class

In [2]:
%load_ext dotenv
%dotenv

In [4]:
import os

In [8]:
from langchain_groq.chat_models import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_core.runnables import RunnablePassthrough

In [9]:
RunnablePassthrough().invoke("Hii")

'Hii'

In [10]:
list_output_parser = CommaSeparatedListOutputParser()

In [28]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job_title} needs?
''' + list_output_parser.get_format_instructions())

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy to master all the tools:
{tools}
''')

In [29]:
chat_template_tools

ChatPromptTemplate(input_variables=['job_title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job_title'], input_types={}, partial_variables={}, template='\nWhat are the five most important tools a {job_title} needs?\nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'), additional_kwargs={})])

In [30]:
str_output_parser = StrOutputParser()

In [31]:
llm = ChatGroq(
    model='openai/gpt-oss-120b',
    api_key=os.environ.get('GROQ_API_KEY'),
    max_tokens=1500
)

In [32]:
chain_tools = chat_template_tools | llm | list_output_parser

In [33]:
chain_strategy = chat_template_strategy | llm | str_output_parser

In [34]:
chain_combine = chain_tools | chain_strategy

In [35]:
response = chain_combine.invoke(
    {'job_title': 'ML Engineer'}
)

In [36]:
print(response)

## Master‑Tool Blueprint  
**Goal:** Become proficient (able to design, implement, and ship production‑ready solutions) with **Python, TensorFlow, PyTorch, Git, and Docker** in a systematic, integrated way.  

Below is a **step‑by‑step strategy** that blends theory, hands‑on practice, and real‑world projects. Adjust the pacing to your available hours, but the structure works well for a **10‑hour/week** commitment (≈ 6‑month plan) or an **20‑hour/week** sprint (≈ 3‑month plan).

---

## 1️⃣ Foundations – “The Core”

| Week | Focus | Resources | Milestones |
|------|-------|-----------|------------|
| 1‑2 | **Python fundamentals** – data structures, functions, OOP, modules, virtual environments | • *Automate the Boring Stuff* (free)  <br>• *Python Crash Course* <br>• Official docs (PEP‑8) <br>• Real‑Python tutorials (list comprehensions, generators) | • Write 5 scripts that solve everyday problems (CSV parsing, API call, simple CLI). <br>• Set up a `venv` + `pip` workflow. |
| 3‑4 | **Py

## Graphing Runnables

In [37]:
chain_combine.get_graph().print_ascii()

          +-------------+          
          | PromptInput |          
          +-------------+          
                 *                 
                 *                 
                 *                 
      +--------------------+       
      | ChatPromptTemplate |       
      +--------------------+       
                 *                 
                 *                 
                 *                 
           +----------+            
           | ChatGroq |            
           +----------+            
                 *                 
                 *                 
                 *                 
+--------------------------------+ 
| CommaSeparatedListOutputParser | 
+--------------------------------+ 
                 *                 
                 *                 
                 *                 
      +--------------------+       
      | ChatPromptTemplate |       
      +--------------------+       
                 *          

## RunnableParallel

In [38]:
from langchain_core.runnables import RunnableParallel

In [41]:
chain_parallel = RunnableParallel({'job_title': chain_tools, 'tools': chain_strategy})

In [43]:
chain_parallel.invoke({'job_title': 'Data Scientist', 'tools': ['python', 'docker']})

{'job_title': ['Python', 'Jupyter Notebook', 'SQL', 'Git', 'TensorFlow'],
 'tools': '## Mastering **Python**\u202f+\u202f**Docker** – A Combined Learning Road‑Map  \n\nBelow is a step‑by‑step strategy that lets you become **proficient** in each tool **individually** and then **integrate** them so you can build, ship, and run Python applications reliably in containers.  \nThe plan is broken into **four phases** (Foundations, Intermediate, Advanced Integration, Real‑World Mastery) with **weekly milestones**, **key resources**, **hands‑on projects**, and **assessment checkpoints**. Feel free to adjust the pace to fit your schedule (e.g., 10\u202fh/week → ~4\u202fmonths; 20\u202fh/week → ~2\u202fmonths).\n\n---\n\n## 📚 Phase\u202f1 – Foundations (Weeks\u202f1‑4)\n\n| Week | Python Goal | Docker Goal | Deliverable |\n|------|-------------|-------------|-------------|\n| 1 | Install Python\u202f3.11+, set up a virtual‑env workflow (`venv`, `pip`, `pyproject.toml`). Learn basic syntax, data t

In [44]:
chain_parallel.get_graph().print_ascii()

                   +--------------------------------+                
                   | Parallel<job_title,tools>Input |                
                   +--------------------------------+                
                         ****               ****                     
                      ***                       ***                  
                    **                             **                
      +--------------------+                +--------------------+   
      | ChatPromptTemplate |                | ChatPromptTemplate |   
      +--------------------+                +--------------------+   
                 *                                      *            
                 *                                      *            
                 *                                      *            
           +----------+                           +----------+       
           | ChatGroq |                           | ChatGroq |       
           +--------